In [12]:
# load env variables and create client
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()

model = "deepseek-chat"
base_url = "https://api.deepseek.com"

messages = []

client = OpenAI(
    api_key = os.getenv("DEEPSEEK_API_KEY"),
    base_url = base_url
)

def add_user_message(messages, text):
    messages.append({
        "role": "user",
        "content": text
    })
    return messages

def add_assistant_message(messages, text):
    messages.append({
        "role": "assistant",
        "content": text
    })

def chat(messages, system_prompt=None, temprature=1.0, stop_sequences=None):
    if stop_sequences is None:
        stop_sequences = []
    if system_prompt: 
        request_messages = [
            {
                "role": "system",
                "content": system_prompt
            },
            *messages
        ]
    else:
        request_messages = messages
    response.client.chat.completion.create(
        model = model,
        messages = request_messages,
        temprature = temprature,
        stop = stop_sequences
    )
    answer = response.choices[0].message.content
    add_assistant_message(message, answer)
    return answer

In [19]:
from datetime import datetime

def get_current_datetime():
    return datetime.now().strftime("%A, %B %d, %Y %I:%M:%S %p")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_datetime",
            "description": "Returns the current date and time.",
            "parameters": {
                "type": "object",
                "properties": {},
                "required": []
            }
        }
    }
]    

In [20]:
# call tool
request_messages = [
    {
        "role": "user",
        "content": "What is the current date and time?"
    }
]

response = client.chat.completions.create(
    model=model,
    messages=request_messages,
    temperature=1.0,
    tools=tools,
    tool_choice="auto"
)

In [22]:
from datetime import datetime

def get_current_datetime():
    return datetime.now().strftime("%A, %B %d, %Y %I:%M:%S %p")

tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_datetime",
            "description": "Returns the current date and time.",
            "parameters": {
                "type": "object",
                "properties": {}
            }
        }
    }
]

request_messages = [
    {
        "role": "user",
        "content": "What is the current date and time?"
    }
]

response = client.chat.completions.create(
    model=model,
    messages=request_messages,
    temperature=1.0,
    tools=tools,
    tool_choice="auto"
)

message = response.choices[0].message

if message.tool_calls:

    for tool_call in message.tool_calls:

        if tool_call.function.name == "get_current_datetime":

            result = get_current_datetime()

            request_messages.append(message)

            request_messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": result
            })

    response = client.chat.completions.create(
        model=model,
        messages=request_messages
    )

    # print(response.choices[0].message.content)
    print(response)

ChatCompletion(id='f5e1ef2b-9f5a-4283-bb5e-00bca3baaee8', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='The current date and time is **Sunday, October 25, 2026 at 8:54:48 PM**.', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None))], created=1792941889, model='deepseek-flash', object='chat.completion', metadata=None, moderation=None, service_tier=None, system_fingerprint='e72d3c7f663d3e1383b00b3884cb4501', usage=CompletionUsage(completion_tokens=25, prompt_tokens=73, total_tokens=98, completion_tokens_details=None, prompt_tokens_details=PromptTokensDetails(audio_tokens=None, cache_write_tokens=None, cached_tokens=0, image_tokens=None, text_tokens=None), prompt_cache_hit_tokens=0, prompt_cache_miss_tokens=73))
